In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class CRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # keeping the layers we want from the CNN
        self.cnn = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        # first iteration we're freezing CNN and only training RNN and CTC head
        for param in self.cnn.parameters():
            param.requires_grad = False

        # use this to force height of images down to be 1
        self.height_pool = nn.AdaptiveAvgPool2d((1, None))

        # use Bi-LSTM as the RNN
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        # input to linear layer is twice the hidden size since using bidirectional LSTM
        self.fc = nn.Linear(256*2, vocab_size)

    def forward(self, x):
        features = self.cnn(x)
        # collapse everything down to height of 1
        # allows use of images from different datasets without worrying about different heights
        features = self.height_pool(features)
        # remove the height dimension to go to 2D
        features = features.squeeze(2)
        # reorder the tensors so they're in format to be fed into RNN
        # from documentation it needs to be in format of (N,L,Hin) and output of cnn was (B, C, H, W)
        # after removing height dimension features is of format (B, C, W)
            # N is batch size (B)
            # L is sequence length (W)
            # Hin is input length (C)
        features = features.permute(0,2,1)  # (B,W,C)

        seq, _ = self.rnn(features)
        out = self.fc(seq)

        return out

CHARS = " !\"'()*+,-./:;?abcdefghijklmnopqrstuvwxyz0123456789"

def get_transforms(img_height: int = 32, augment: bool = False):
    # base pipeline used for val and test with no augmentation
    base = [
        ResizeKeepAspect(img_height),
        transforms.ToTensor(),                        # PIL -> tensor, scales to [0, 1]
        transforms.Normalize(mean=[0.5], std=[0.5]),  # re-center to [-1, 1]
    ]
    if augment:
        # training pipeline adds warp and blur to simulate real world handwriting photos
        base = [
            ResizeKeepAspect(img_height),
            transforms.RandomAffine(degrees=2, translate=(0.02, 0.02)),  # slight tilt/shift
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),    # simulate blurry images
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ]
    return transforms.Compose(base)

class Vocab:
    def __init__(self, chars: str = CHARS):
        # blank token at index 0 (required by CTC)
        self.blank_idx = 0
        # shift all real characters up by 1 to leave index 0 free for blank
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i + 1: ch for i, ch in enumerate(chars)}
        self.size = len(chars) + 1  # +1 for blank

    def encode(self, text: str) -> list[int]:
        text = text.lower()
        # skip any characters not in vocab rather than crashing
        return [self.char2idx[ch] for ch in text if ch in self.char2idx]
    
    def decode(self, indices: list[int], remove_duplicates=True, remove_blank=True) -> str:
        result = []
        prev = None
        for idx in indices:
            # collapse consecutive duplicates (raw CTC output holds characters across many timesteps)
            if remove_duplicates and idx == prev:
                prev = idx
                continue
            prev = idx
            # strip blank tokens
            if remove_blank and idx == self.blank_idx:
                continue
            if idx in self.idx2char:
                result.append(self.idx2char[idx])
        return "".join(result)
    
class ResizeKeepAspect:
    def __init__(self, height: int):
        self.height = height

    def __call__(self, img: Image.Image) -> Image.Image:
        # grayscale first to drop color noise, then back to RGB for ResNet18
        img = img.convert("L").convert("RGB")
        w, h = img.size
        # scale width proportionally to preserve aspect ratio
        new_w = max(1, int(w * self.height / h))  # max(1) guards against rounding to 0
        return img.resize((new_w, self.height), Image.BILINEAR)
    
class IAMLineDataset(Dataset):
    def __init__(self, split: str, vocab: Vocab, transform=None):
        print(f"Loading IAM-line [{split}] from HuggingFace")
        # HuggingFace loads lazily so this doesn't load all images into memory at once
        self.data = load_dataset("Teklia/IAM-line", split=split)
        self.vocab = vocab
        self.transform = transform
        print(f"  -> {len(self.data)} samples loaded.")

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["image"]
        text = item["text"].lower()

        if self.transform:
            image = self.transform(image)  # PIL image -> (3, 32, W) tensor

        # dtype=long required specifically by nn.CTCLoss for target labels
        label = torch.tensor(self.vocab.encode(text), dtype=torch.long)
        # return raw text too so eval can compute CER/WER without decoding the tensor
        return image, label, text

def collate_fn(batch):
    # batch is a list of (image, label, text) tuples - unzip into three separate tuples
    images, labels, texts = zip(*batch)

    # images have the same height (32) but different widths - pad to max width in this batch
    max_w = max(img.shape[2] for img in images)
    padded = []
    for img in images:
        pad_w = max_w - img.shape[2]
        # pad right side only, value=-1.0 is black in normalized range and safely out of real image range
        padded.append(torch.nn.functional.pad(img, (0, pad_w), value=-1.0))
    images_tensor = torch.stack(padded)  # (B, 3, 32, W_max)

    # CTC loss requires labels as a single flat 1D tensor, not a padded 2D matrix
    labels_concat = torch.cat(labels)

    # CTC also needs to know where each label ends since they're all concatenated together
    label_lens = torch.tensor([len(l) for l in labels], dtype=torch.long)

    # CNN downsamples width by factor of 32 (ResNet18's total stride)
    # CTC needs this to know the sequence length for each sample
    input_lens = torch.tensor([img.shape[2] // 32 for img in images], dtype=torch.long)

    return images_tensor, labels_concat, label_lens, input_lens, texts

def get_dataloaders(vocab: Vocab, img_height: int = 32, batch_size: int = 32):
    train_ds = IAMLineDataset("train",      vocab, transform=get_transforms(img_height, augment=True))
    val_ds   = IAMLineDataset("validation", vocab, transform=get_transforms(img_height, augment=False))
    test_ds  = IAMLineDataset("test",       vocab, transform=get_transforms(img_height, augment=False))

    # shuffle training data each epoch, no need to shuffle val/test
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader